# Household Object Recognition — cups, bottles, keys, pens

**Computer Vision course project.** Four-class image classification on a self-collected dataset.

| Step | Section |
|---|---|
| Dataset collection protocol and integrity checks | 1–2 |
| Exploratory data analysis | 3 |
| Stratified train / val / test split | 4 |
| Preprocessing and data augmentation | 5 |
| Baselines: HOG + colour features → Logistic Regression, SVM; simple CNN | 6–7 |
| Improved models: transfer learning (ResNet-18, EfficientNet-B0) | 8 |
| Evaluation: accuracy, precision, recall, F1, confusion matrices, error analysis, Grad-CAM | 9–10 |
| Augmentation ablation, inference on new photos, export of results for the report | 11–13 |

**How to run.** Upload the dataset folder (or the zip) to Google Drive, open this notebook in Google Colab with a GPU runtime (*Runtime → Change runtime type → T4 GPU*), set `DATA_ROOT` in the config cell and run all cells. Every figure and table used in the report is saved to `results/`.

## 0. Setup

In [ ]:
# In Colab most packages are pre-installed; this only adds what may be missing.
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import skimage  # noqa
except ImportError:
    _pip("scikit-image")
try:
    import pillow_heif  # reads iPhone .HEIC photos
    pillow_heif.register_heif_opener()
except ImportError:
    _pip("pillow-heif")
    try:
        import pillow_heif; pillow_heif.register_heif_opener()
    except Exception:
        pass

In [ ]:
# Optional: mount Google Drive when running in Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import os, json, time, random, hashlib, shutil, copy, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from skimage.feature import hog

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
sns.set_style("whitegrid")

### Configuration

`SMOKE_TEST` is only for checking that the code runs (tiny epochs, no pretrained weights). Keep it `False` for real experiments.

In [ ]:
# ---------------- paths ----------------
DATA_ROOT = Path(os.environ.get("CV_DATA_ROOT", "/content/drive/MyDrive/household_objects"))
RAW_DIR   = DATA_ROOT / "raw"        # raw/<class_name>/*.jpg  — your photos
PROC_DIR  = DATA_ROOT / "processed"  # created by section 2 (cleaned + resized copies)
OUT_DIR   = Path(os.environ.get("CV_OUT_DIR", DATA_ROOT / "results"))  # saved to Drive, survives the Colab session
assert RAW_DIR.exists(), f"Folder not found: {RAW_DIR}. Check the folder name/location in Google Drive."
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- task ----------------
CLASSES   = ["bottle", "cup", "key", "pen"]
NUM_CLASSES = len(CLASSES)
IMG_SIZE  = 224          # network input size
STORE_SIZE = 320         # short side of stored processed images (> IMG_SIZE leaves room for random crops)
SEED      = 42
SPLIT     = (0.70, 0.15, 0.15)   # train / val / test
MAX_PER_CLASS = 30               # balance classes: keep at most N photos per class (None = keep all)

# ---------------- training ----------------
SMOKE_TEST = os.environ.get("CV_SMOKE_TEST", "0") == "1"
PRETRAINED = not SMOKE_TEST
BATCH_SIZE = 8 if SMOKE_TEST else 32
EPOCHS_CNN = 3 if SMOKE_TEST else 40          # simple CNN from scratch
EPOCHS_HEAD = 1 if SMOKE_TEST else 5          # transfer learning, stage 1 (frozen backbone)
EPOCHS_FT   = 2 if SMOKE_TEST else 15         # transfer learning, stage 2 (fine-tuning)
PATIENCE    = 10
NUM_WORKERS = 0 if SMOKE_TEST else 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()
print("Device:", DEVICE, "| torch", torch.__version__, "| smoke test:", SMOKE_TEST)

## 1. Dataset collection protocol

The dataset is collected by the author with a phone camera. To make it **diverse** (20 % of the grade), each class is photographed under deliberately varied conditions:

| Factor | Variation used |
|---|---|
| Object instances | ≥ 8 different physical objects per class (e.g. glass / ceramic / paper cups; plastic / glass / metal bottles; door / car / padlock keys; ballpoint / gel / marker pens) |
| Background | desk, kitchen table, bed / fabric, floor, hand-held, cluttered shelf |
| Lighting | daylight near window, warm indoor lamp, cold LED, dim / evening |
| Viewpoint | top-down, side, 45°, close-up, far away; object rotated |
| Scale & position | object filling 20 %–90 % of the frame, not always centred |
| Occlusion / clutter | partially covered by hand, other objects present (but the labelled object dominates) |

Target: **150–250 photos per class** (minimum required: 100–200), balanced across classes.
Rules: one dominant object per photo; no near-identical burst shots; photos taken on different days.

Folder layout expected by the notebook:
```
household_objects/
└── raw/
    ├── bottle/  IMG_0001.jpg ...
    ├── cup/
    ├── key/
    └── pen/
```
A metadata sheet (`metadata.csv`: filename, class, instance_id, background, lighting) is optional but very useful for the report and for a paper. If it exists in `DATA_ROOT`, it is used in the EDA.

**Optional helper:** the next cell captures photos from a laptop webcam (local Jupyter only, not Colab). Press **space** to save, **n** to switch class, **q** to quit.

In [ ]:
def capture_with_webcam(out_dir=RAW_DIR, classes=CLASSES, cam_index=0):
    import cv2
    cap = cv2.VideoCapture(cam_index)
    ci = 0
    for c in classes: (Path(out_dir) / c).mkdir(parents=True, exist_ok=True)
    while True:
        ok, frame = cap.read()
        if not ok: break
        cls = classes[ci]
        n = len(list((Path(out_dir) / cls).glob("*.jpg")))
        view = frame.copy()
        cv2.putText(view, f"{cls}: {n}  [space]=save [n]=next class [q]=quit", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.imshow("capture", view)
        k = cv2.waitKey(1) & 0xFF
        if k == ord(" "):
            cv2.imwrite(str(Path(out_dir) / cls / f"{cls}_{int(time.time()*1000)}.jpg"), frame)
        elif k == ord("n"):
            ci = (ci + 1) % len(classes)
        elif k == ord("q"):
            break
    cap.release(); cv2.destroyAllWindows()

# capture_with_webcam()   # uncomment to use

## 2. Data cleaning and integrity checks

For every raw image we:
1. check that it opens (corrupted files are reported and skipped);
2. apply the **EXIF orientation** (phone photos are often stored rotated);
3. convert to RGB (drops alpha, handles HEIC/PNG/grayscale);
4. compute a **perceptual hash (dHash)** to find exact and near-duplicates — duplicates across train/test would inflate the scores;
5. save a resized copy (short side = `STORE_SIZE`) to `processed/` — the raw 12 MP photos are far too large to decode at every epoch.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic", ".heif"}

def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def hamming(a, b):
    return bin(a ^ b).count("1")

def resize_short(img, short=STORE_SIZE):
    w, h = img.size
    s = short / min(w, h)
    if s >= 1: return img
    return img.resize((round(w * s), round(h * s)), Image.LANCZOS)

records, bad = [], []
for cls in CLASSES:
    files = sorted(p for p in (RAW_DIR / cls).rglob("*") if p.suffix.lower() in IMG_EXT)
    (PROC_DIR / cls).mkdir(parents=True, exist_ok=True)
    for p in files:
        try:
            im = Image.open(p); im = ImageOps.exif_transpose(im).convert("RGB")
        except (UnidentifiedImageError, OSError) as e:
            bad.append((str(p), str(e))); continue
        w, h = im.size
        out = PROC_DIR / cls / (p.stem + ".jpg")
        if not out.exists():
            out.parent.mkdir(parents=True, exist_ok=True)   # Drive can drop freshly deleted/created folders
            resize_short(im).save(out, quality=95)
        records.append(dict(path=str(out), raw_path=str(p), label=cls, width=w, height=h,
                            hash=dhash(im), bytes=p.stat().st_size))

df = pd.DataFrame(records)
print(f"Readable images: {len(df)} | corrupted / unreadable: {len(bad)}")
for b in bad[:10]: print("  BAD:", b)
df.groupby("label").size().rename("images").to_frame().T

In [ ]:
# Near-duplicate detection (Hamming distance of dHash <= 4 bits out of 64)
DUP_THRESHOLD = 4
hashes = df["hash"].values
dup_pairs = []
for i in range(len(hashes)):
    d = np.array([hamming(int(hashes[i]), int(h)) for h in hashes[i + 1:]])
    for j in np.where(d <= DUP_THRESHOLD)[0]:
        dup_pairs.append((i, i + 1 + j, int(d[j])))
print(f"Near-duplicate pairs found: {len(dup_pairs)}")

# Keep the first image of each duplicate group, drop the rest
drop = {j for _, j, _ in dup_pairs}
df_dups = df.iloc[sorted(drop)] if drop else df.iloc[0:0]
df = df.drop(index=sorted(drop)).reset_index(drop=True)
print(f"Images after de-duplication: {len(df)}")

# Balance classes without deleting anything on Drive: randomly keep at most MAX_PER_CLASS photos per class
if MAX_PER_CLASS:
    keep = [i for _, g in df.groupby("label")
              for i in g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED).index]
    df = df.loc[sorted(keep)].reset_index(drop=True)
    print(f"Images after balancing (max {MAX_PER_CLASS} per class): {len(df)}")
    print(df["label"].value_counts().reindex(CLASSES).to_dict())

if dup_pairs:
    show = dup_pairs[:4]
    fig, ax = plt.subplots(len(show), 2, figsize=(5, 2.5 * len(show)), squeeze=False)
    for r, (i, j, d) in enumerate(show):
        for c, k in enumerate((i, j)):
            ax[r, c].imshow(Image.open(records[k]["path"])); ax[r, c].axis("off")
            ax[r, c].set_title(f"{records[k]['label']} (d={d})", fontsize=8)
    plt.suptitle("Examples of removed near-duplicates"); plt.show()

df.drop(columns=["hash"]).to_csv(OUT_DIR / "dataset_index.csv", index=False)

## 3. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
counts = df["label"].value_counts().reindex(CLASSES)
sns.barplot(x=counts.index, y=counts.values, ax=ax[0], palette="Set2")
for i, v in enumerate(counts.values): ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[0].set_title("Images per class"); ax[0].set_ylabel("count")

sns.scatterplot(data=df, x="width", y="height", hue="label", ax=ax[1], s=15, hue_order=CLASSES)
ax[1].set_title("Original resolution (px)")

df["aspect"] = df["width"] / df["height"]
sns.histplot(data=df, x="aspect", hue="label", ax=ax[2], bins=20, element="step", hue_order=CLASSES)
ax[2].set_title("Aspect ratio (w / h)")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_eda.png"); plt.show()

imb = counts.max() / counts.min()
print(f"Class imbalance ratio (max/min): {imb:.2f}")

In [ ]:
def show_grid(frame, n_per_class=6, fname=None, title="Sample images per class"):
    fig, ax = plt.subplots(NUM_CLASSES, n_per_class, figsize=(2 * n_per_class, 2.1 * NUM_CLASSES))
    for r, cls in enumerate(CLASSES):
        sub = frame[frame.label == cls]
        sample = sub.sample(min(n_per_class, len(sub)), random_state=SEED)
        for c in range(n_per_class):
            ax[r, c].axis("off")
            if c < len(sample):
                ax[r, c].imshow(Image.open(sample.iloc[c]["path"]))
        ax[r, 0].set_title(cls, loc="left", fontsize=11, fontweight="bold")
    plt.suptitle(title); plt.tight_layout()
    if fname: plt.savefig(OUT_DIR / fname)
    plt.show()

show_grid(df, fname="fig_samples.png")

In [ ]:
# Colour statistics per class (mean RGB of each image) — shows whether colour alone could separate classes
def mean_rgb(p):
    a = np.asarray(Image.open(p).resize((64, 64)), dtype=np.float32) / 255
    return a.reshape(-1, 3).mean(0)
rgb = np.stack([mean_rgb(p) for p in df["path"]])
df[["mean_r", "mean_g", "mean_b"]] = rgb
df["brightness"] = rgb.mean(1)
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=df, x="label", y="brightness", order=CLASSES, palette="Set2", ax=ax)
ax.set_title("Image brightness per class"); plt.savefig(OUT_DIR / "fig_brightness.png"); plt.show()

# Optional metadata (background / lighting) — useful for the dataset-diversity section of the report
meta_path = DATA_ROOT / "metadata.csv"
if meta_path.exists():
    meta = pd.read_csv(meta_path)
    df["filename"] = df["raw_path"].map(lambda s: Path(s).name)
    df = df.merge(meta.drop(columns=[c for c in ["class", "label"] if c in meta.columns]),
                  on="filename", how="left")
    for col in ["background", "lighting"]:
        if col in df.columns:
            display(pd.crosstab(df["label"], df[col]))

## 4. Train / validation / test split

Stratified split 70 / 15 / 15 with a fixed seed. The test set is **never** used for model selection or early stopping — only the validation set is. The split is saved to `results/split.csv` so every model is evaluated on exactly the same images.

> If `metadata.csv` contains an `instance_id` column (which physical object is in the photo), the split is done **by instance** — all photos of the same cup go into the same subset. This is a stricter and more honest test of generalisation to *unseen objects*, and is worth reporting in a paper.

In [ ]:
label2idx = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["label"].map(label2idx)

if "instance_id" in df.columns and df["instance_id"].notna().all():
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=SPLIT[2], random_state=SEED)
    trv_idx, te_idx = next(gss.split(df, df["y"], groups=df["instance_id"]))
    trv = df.iloc[trv_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=SPLIT[1] / (SPLIT[0] + SPLIT[1]), random_state=SEED)
    tr_idx, va_idx = next(gss2.split(trv, trv["y"], groups=trv["instance_id"]))
    train_df, val_df, test_df = trv.iloc[tr_idx], trv.iloc[va_idx], df.iloc[te_idx]
    print("Split mode: by object instance")
else:
    train_df, tmp = train_test_split(df, test_size=SPLIT[1] + SPLIT[2], stratify=df["y"], random_state=SEED)
    val_df, test_df = train_test_split(tmp, test_size=SPLIT[2] / (SPLIT[1] + SPLIT[2]),
                                       stratify=tmp["y"], random_state=SEED)
    print("Split mode: stratified random")

train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
split_df = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")], ignore_index=True)
split_df[["path", "label", "split"]].to_csv(OUT_DIR / "split.csv", index=False)
split_table = pd.crosstab(split_df["label"], split_df["split"])[["train", "val", "test"]]
split_table.loc["total"] = split_table.sum()
split_table

## 5. Preprocessing and data augmentation

**Preprocessing (all models):** EXIF fix + RGB (section 2) → resize → centre crop to 224×224 → scale to [0, 1] → normalise with ImageNet mean/std (required by pretrained backbones; used for the CNN from scratch too, for a fair comparison).

**Augmentation (training set only, applied on-the-fly so every epoch sees new variants):**

| Transform | Parameters | Reason |
|---|---|---|
| RandomResizedCrop | scale 0.6–1.0, ratio 0.75–1.33 | object size / position varies |
| RandomHorizontalFlip | p = 0.5 | objects are left-right symmetric in meaning |
| RandomRotation | ±20° | phone held at different angles; pens and keys lie in any orientation |
| RandomPerspective | distortion 0.2, p = 0.3 | different viewpoints |
| ColorJitter | brightness/contrast/saturation 0.3, hue 0.05 | different lighting and white balance |
| RandomGrayscale | p = 0.1 | stops the model relying only on colour |
| GaussianBlur | p = 0.2 | slight motion / focus blur from phone |
| RandomErasing | p = 0.25 | partial occlusion (fingers, other objects) |

Vertical flip is *not* used: cups and bottles have a natural "up".

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
noaug_tf = eval_tf   # used in the augmentation ablation (section 11)

class ImageDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.y, self.tf = frame["path"].tolist(), frame["y"].tolist(), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.y[i]

def make_loader(frame, tf, shuffle):
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ImageDataset(frame, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", generator=g)

train_loader = make_loader(train_df, train_tf, True)
val_loader   = make_loader(val_df, eval_tf, False)
test_loader  = make_loader(test_df, eval_tf, False)

def denorm(t):
    return (t.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)

In [ ]:
# Visualise augmentation: original + 7 random augmented versions of the same photo, for each class
fig, ax = plt.subplots(NUM_CLASSES, 8, figsize=(16, 2.2 * NUM_CLASSES))
for r, cls in enumerate(CLASSES):
    p = train_df[train_df.label == cls].iloc[0]["path"]
    im = Image.open(p).convert("RGB")
    ax[r, 0].imshow(denorm(eval_tf(im))); ax[r, 0].set_title(f"{cls}: original", fontsize=9)
    for c in range(1, 8):
        ax[r, c].imshow(denorm(train_tf(im)))
    for a in ax[r]: a.axis("off")
plt.suptitle("Data augmentation examples"); plt.tight_layout()
plt.savefig(OUT_DIR / "fig_augmentation.png"); plt.show()

## 6. Common evaluation utilities
Metrics are computed on the **test set**. Precision, recall and F1 are reported per class and as **macro averages** (each class weighted equally).

In [ ]:
RESULTS = {}   # model name -> dict of metrics, predictions, timings

def compute_metrics(name, y_true, y_pred, extra=None):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    res = dict(model=name, accuracy=acc, precision=p, recall=r, f1=f1,
               y_true=np.asarray(y_true), y_pred=np.asarray(y_pred))
    if extra: res.update(extra)
    RESULTS[name] = res
    print(f"=== {name} ===  acc={acc:.4f}  macro-P={p:.4f}  macro-R={r:.4f}  macro-F1={f1:.4f}")
    print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES),
                                target_names=CLASSES, digits=4, zero_division=0))
    return res

def plot_confusion(name, ax=None, normalize=True):
    res = RESULTS[name]
    cm = confusion_matrix(res["y_true"], res["y_pred"], labels=range(NUM_CLASSES))
    own = ax is None
    if own: fig, ax = plt.subplots(figsize=(4.4, 3.8))
    if normalize:
        cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
        annot = np.array([[f"{cmn[i,j]:.2f}\n({cm[i,j]})" for j in range(NUM_CLASSES)] for i in range(NUM_CLASSES)])
        sns.heatmap(cmn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                    xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    else:
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(name, fontsize=10)
    if own:
        plt.savefig(OUT_DIR / f"cm_{name.replace(' ', '_').replace('+','')}.png"); plt.show()

## 7. Baseline models

### 7.1 Classical ML: hand-crafted features + Logistic Regression / SVM
Each image → 128×128 → feature vector made of
* **HOG** (Histogram of Oriented Gradients; 9 orientations, 16×16 cells, 2×2 blocks) on the grayscale image — describes shape/edges;
* **HSV colour histogram** (16 bins per channel) — describes colour.

Features are standardised and reduced with PCA (95 % variance). Hyper-parameters are chosen with 5-fold stratified cross-validation on the training set only.

In [ ]:
FEAT_SIZE = 128
def handcrafted_features(path):
    im = Image.open(path).convert("RGB")
    im = ImageOps.fit(im, (FEAT_SIZE, FEAT_SIZE), Image.BILINEAR)
    gray = np.asarray(im.convert("L"), dtype=np.float32) / 255
    h = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")
    hsv = np.asarray(im.convert("HSV"))
    col = np.concatenate([np.histogram(hsv[..., k], bins=16, range=(0, 255), density=True)[0] for k in range(3)])
    return np.concatenate([h, col * 10])

t0 = time.time()
X_tr = np.stack([handcrafted_features(p) for p in train_df.path]); y_tr = train_df.y.values
X_va = np.stack([handcrafted_features(p) for p in val_df.path]);   y_va = val_df.y.values
X_te = np.stack([handcrafted_features(p) for p in test_df.path]);  y_te = test_df.y.values
print(f"Feature dim: {X_tr.shape[1]}  | extraction time {time.time() - t0:.1f}s")

# train + val together for CV-based model selection (classical models don't need early stopping)
X_trv, y_trv = np.vstack([X_tr, X_va]), np.concatenate([y_tr, y_va])
cv = StratifiedKFold(n_splits=3 if SMOKE_TEST else 5, shuffle=True, random_state=SEED)

In [ ]:
classical = {
    "HOG+LogReg": (LogisticRegression(max_iter=5000), {"clf__C": [0.01, 0.1, 1, 10]}),
    "HOG+SVM":    (SVC(kernel="rbf"), {"clf__C": [1, 10, 100], "clf__gamma": ["scale", 1e-3, 1e-4]}),
}
for name, (clf, grid) in classical.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("pca", PCA(n_components=0.95, random_state=SEED)), ("clf", clf)])
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="f1_macro", n_jobs=-1)
    t0 = time.time(); gs.fit(X_trv, y_trv); train_time = time.time() - t0
    t0 = time.time(); pred = gs.predict(X_te); inf_ms = (time.time() - t0) / len(X_te) * 1000
    print(f"{name}: best params {gs.best_params_}, CV macro-F1 {gs.best_score_:.4f}")
    compute_metrics(name, y_te, pred, dict(train_time_s=train_time, infer_ms=inf_ms, params=None,
                                           best_params=str(gs.best_params_)))

### 7.2 Simple CNN trained from scratch
Four convolutional blocks (Conv–BN–ReLU–MaxPool; 32→64→128→256 filters), global average pooling, dropout 0.5 and a linear classifier (~1.17 M parameters).

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, n_classes))
    def forward(self, x): return self.head(self.features(x))

def n_params(m): return sum(p.numel() for p in m.parameters())
print(f"SimpleCNN parameters: {n_params(SimpleCNN()):,}")

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); ys, ps, probs = [], [], []
    for x, y in loader:
        out = model(x.to(DEVICE)); pr = out.softmax(1).cpu()
        ys.append(y); ps.append(pr.argmax(1)); probs.append(pr)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(probs).numpy()

def run_epochs(model, optimizer, scheduler, epochs, train_loader, val_loader, history, tag, patience=PATIENCE):
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_f1, best_state, bad_epochs = -1, None, 0
    for ep in range(epochs):
        model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(); out = model(x); loss = crit(out, y)
            loss.backward(); optimizer.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        if scheduler: scheduler.step()
        # validation
        model.eval(); vl, vn = 0.0, 0; yv, pv = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE); out = model(x)
                vl += crit(out, y).item() * len(y); vn += len(y)
                yv.append(y.cpu()); pv.append(out.argmax(1).cpu())
        yv, pv = torch.cat(yv).numpy(), torch.cat(pv).numpy()
        vf1 = precision_recall_fscore_support(yv, pv, average="macro", zero_division=0)[2]
        history.append(dict(stage=tag, train_loss=tl / tn, train_acc=tc / tn, val_loss=vl / vn,
                            val_acc=(yv == pv).mean(), val_f1=vf1))
        h = history[-1]
        print(f"[{tag}] ep {len(history):3d}  train loss {h['train_loss']:.3f} acc {h['train_acc']:.3f} | "
              f"val loss {h['val_loss']:.3f} acc {h['val_acc']:.3f} F1 {vf1:.3f}")
        if vf1 > best_f1:
            best_f1, best_state, bad_epochs = vf1, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping after {bad_epochs} epochs without improvement"); break
    model.load_state_dict(best_state)
    return best_f1

def finish(name, model, history, train_time):
    y, p, prob = predict(model, test_loader)
    # inference speed: single image, averaged
    x1 = next(iter(test_loader))[0][:1].to(DEVICE); model.eval()
    with torch.no_grad():
        for _ in range(3): model(x1)
        t0 = time.time()
        for _ in range(20): model(x1)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    inf_ms = (time.time() - t0) / 20 * 1000
    compute_metrics(name, y, p, dict(train_time_s=train_time, infer_ms=inf_ms, params=n_params(model),
                                     history=pd.DataFrame(history), probs=prob))
    torch.save(model.state_dict(), OUT_DIR / f"{name.replace(' ', '_')}.pt")

In [ ]:
def train_simple_cnn(name, train_loader_):
    seed_everything()
    model = SimpleCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_CNN)
    hist = []; t0 = time.time()
    run_epochs(model, opt, sch, EPOCHS_CNN, train_loader_, val_loader, hist, "scratch", patience=PATIENCE + 5)
    finish(name, model, hist, time.time() - t0)
    return model

cnn_model = train_simple_cnn("SimpleCNN", train_loader)

## 8. Improved models: transfer learning

Backbones pretrained on ImageNet: **ResNet-18** (11.2 M params) and **EfficientNet-B0** (4.0 M params). The original classifier is replaced by `Dropout(0.3) → Linear(→4)`.

Two-stage training:
1. **Head only** — backbone frozen, lr = 1e-3, `EPOCHS_HEAD` epochs. The new randomly-initialised head is trained without destroying the pretrained features.
2. **Fine-tuning** — all layers unfrozen, lr = 1e-4 for the backbone and 1e-3 for the head, cosine schedule, early stopping on validation macro-F1.

In [ ]:
def build_backbone(arch):
    if arch == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, NUM_CLASSES))
        head = m.fc
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, NUM_CLASSES))
        head = m.classifier
    else:
        raise ValueError(arch)
    return m, head

def train_transfer(name, arch, train_loader_=None):
    train_loader_ = train_loader_ or train_loader
    seed_everything()
    model, head = build_backbone(arch); model.to(DEVICE)
    head_ids = {id(p) for p in head.parameters()}
    hist = []; t0 = time.time()
    # stage 1: frozen backbone
    for p in model.parameters(): p.requires_grad = id(p) in head_ids
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    run_epochs(model, opt, None, EPOCHS_HEAD, train_loader_, val_loader, hist, "head")
    # stage 2: fine-tune everything
    for p in model.parameters(): p.requires_grad = True
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{"params": backbone_params, "lr": 1e-4},
                             {"params": head.parameters(), "lr": 1e-3}], weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FT)
    run_epochs(model, opt, sch, EPOCHS_FT, train_loader_, val_loader, hist, "finetune")
    finish(name, model, hist, time.time() - t0)
    return model

resnet_model = train_transfer("ResNet18-TL", "resnet18")

In [ ]:
effnet_model = train_transfer("EfficientNetB0-TL", "efficientnet_b0")

## 9. Results and comparison

In [ ]:
MAIN_MODELS = ["HOG+LogReg", "HOG+SVM", "SimpleCNN", "ResNet18-TL", "EfficientNetB0-TL"]
def summary_table(names):
    rows = []
    for n in names:
        r = RESULTS[n]
        rows.append({"Model": n, "Accuracy": r["accuracy"], "Precision (macro)": r["precision"],
                     "Recall (macro)": r["recall"], "F1 (macro)": r["f1"],
                     "Params": f"{r['params']/1e6:.2f} M" if r.get("params") else "—",
                     "Train time (s)": r["train_time_s"], "Inference (ms/img)": r["infer_ms"]})
    return pd.DataFrame(rows).set_index("Model")

summary = summary_table(MAIN_MODELS)
summary.to_csv(OUT_DIR / "results_summary.csv")
display(summary.style.format({c: "{:.4f}" for c in ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]}
                             | {"Train time (s)": "{:.1f}", "Inference (ms/img)": "{:.2f}"})
                     .highlight_max(subset=["Accuracy", "F1 (macro)"], color="#c6efce"))

ax = summary[["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]].plot.bar(figsize=(10, 4), rot=0, width=0.8)
ax.set_ylim(0, 1.05); ax.set_title("Test-set metrics by model"); ax.legend(loc="lower right", ncol=4, fontsize=8)
plt.savefig(OUT_DIR / "fig_model_comparison.png"); plt.show()

In [ ]:
# Per-class F1 for every model
per_class = pd.DataFrame({n: precision_recall_fscore_support(RESULTS[n]["y_true"], RESULTS[n]["y_pred"],
                                                             labels=range(NUM_CLASSES), zero_division=0)[2]
                          for n in MAIN_MODELS}, index=CLASSES).T
per_class.to_csv(OUT_DIR / "results_per_class_f1.csv")
display(per_class.style.format("{:.3f}").background_gradient(cmap="RdYlGn", vmin=0.5, vmax=1))

In [ ]:
fig, axes = plt.subplots(1, len(MAIN_MODELS), figsize=(4.2 * len(MAIN_MODELS), 3.8))
for ax, n in zip(axes, MAIN_MODELS): plot_confusion(n, ax=ax)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_confusion_matrices.png"); plt.show()

In [ ]:
# Learning curves for the deep models
deep = ["SimpleCNN", "ResNet18-TL", "EfficientNetB0-TL"]
fig, ax = plt.subplots(2, len(deep), figsize=(5 * len(deep), 6.5))
for c, n in enumerate(deep):
    h = RESULTS[n]["history"]; ep = np.arange(1, len(h) + 1)
    ax[0, c].plot(ep, h.train_loss, label="train"); ax[0, c].plot(ep, h.val_loss, label="val")
    ax[1, c].plot(ep, h.train_acc, label="train"); ax[1, c].plot(ep, h.val_acc, label="val")
    if (h.stage == "finetune").any():
        s = ep[(h.stage == "finetune").values][0]
        for a in ax[:, c]: a.axvline(s - 0.5, ls="--", c="gray", lw=1)
    ax[0, c].set_title(f"{n} — loss"); ax[1, c].set_title(f"{n} — accuracy")
    ax[1, c].set_xlabel("epoch"); ax[0, c].legend(); ax[1, c].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_learning_curves.png"); plt.show()

## 10. Error analysis and interpretability

### 10.1 Misclassified test images (best model)

In [ ]:
best_name = max(["SimpleCNN", "ResNet18-TL", "EfficientNetB0-TL"], key=lambda n: RESULTS[n]["f1"])
best_model = {"SimpleCNN": cnn_model, "ResNet18-TL": resnet_model, "EfficientNetB0-TL": effnet_model}[best_name]
print("Best deep model:", best_name)

r = RESULTS[best_name]
wrong = np.where(r["y_true"] != r["y_pred"])[0]
print(f"Misclassified: {len(wrong)} / {len(r['y_true'])}")
if len(wrong):
    k = min(12, len(wrong)); cols = min(6, k); rows = int(np.ceil(k / cols))
    fig, ax = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.9 * rows), squeeze=False)
    for a in ax.flat: a.axis("off")
    for a, i in zip(ax.flat, wrong[:k]):
        a.imshow(Image.open(test_df.path[i]))
        a.set_title(f"true: {CLASSES[r['y_true'][i]]}\npred: {CLASSES[r['y_pred'][i]]} ({r['probs'][i].max():.2f})",
                    fontsize=8, color="crimson")
    plt.suptitle(f"Misclassified test images — {best_name}"); plt.tight_layout()
    plt.savefig(OUT_DIR / "fig_misclassified.png"); plt.show()

### 10.2 Grad-CAM
Grad-CAM highlights which image regions drove the prediction. It checks that the network looks at the **object** and not at the background (a common shortcut when, e.g., all keys were photographed on the same table).

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0].detach()))
    def __call__(self, x, cls=None):
        self.model.eval(); x = x.requires_grad_(True)
        out = self.model(x); cls = out.argmax(1).item() if cls is None else cls
        self.model.zero_grad(); out[0, cls].backward()
        w = self.grads.mean((2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), cls

def target_layer(name, model):
    if name.startswith("ResNet"): return model.layer4[-1]
    if name.startswith("EfficientNet"): return model.features[-1]
    return model.features[-1]

cam = GradCAM(best_model, target_layer(best_name, best_model))
fig, ax = plt.subplots(2, 2 * NUM_CLASSES, figsize=(2.2 * 2 * NUM_CLASSES, 4.6))
for c, cls in enumerate(CLASSES):
    sub = test_df[test_df.label == cls].head(2)
    for r_, (_, row) in enumerate(sub.iterrows()):
        x = eval_tf(Image.open(row.path).convert("RGB")).unsqueeze(0).to(DEVICE)
        heat, pc = cam(x)
        img = denorm(x[0].detach().cpu())
        ax[r_, 2 * c].imshow(img); ax[r_, 2 * c].set_title(f"true: {cls}", fontsize=8)
        ax[r_, 2 * c + 1].imshow(img); ax[r_, 2 * c + 1].imshow(heat, cmap="jet", alpha=0.45)
        ax[r_, 2 * c + 1].set_title(f"pred: {CLASSES[pc]}", fontsize=8)
for a in ax.flat: a.axis("off")
plt.suptitle(f"Grad-CAM — {best_name}"); plt.tight_layout()
plt.savefig(OUT_DIR / "fig_gradcam.png"); plt.show()

## 11. Ablation: effect of data augmentation
The simple CNN and ResNet-18 are retrained **without augmentation** (only resize + crop + normalise) on the same split. The difference in test macro-F1 quantifies how much augmentation helps on a small dataset.

In [ ]:
train_loader_noaug = make_loader(train_df, noaug_tf, True)
_ = train_simple_cnn("SimpleCNN (no aug)", train_loader_noaug)
_ = train_transfer("ResNet18-TL (no aug)", "resnet18", train_loader_noaug)

ablation = pd.DataFrame([
    {"Model": base, "With augmentation": RESULTS[base]["f1"], "Without augmentation": RESULTS[f"{base} (no aug)"]["f1"]}
    for base in ["SimpleCNN", "ResNet18-TL"]]).set_index("Model")
ablation["Δ F1"] = ablation["With augmentation"] - ablation["Without augmentation"]
ablation.to_csv(OUT_DIR / "results_ablation_augmentation.csv")
ablation.style.format("{:.4f}")

## 12. Inference on new photos
Put any new photos in a folder (or give a single path) — the best model predicts the class with its confidence.

In [ ]:
@torch.no_grad()
def classify(path, model=best_model, topk=3):
    model.eval()
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    prob = model(eval_tf(im).unsqueeze(0).to(DEVICE)).softmax(1)[0].cpu().numpy()
    order = prob.argsort()[::-1][:topk]
    plt.figure(figsize=(3, 3)); plt.imshow(im); plt.axis("off")
    plt.title(f"{CLASSES[order[0]]} ({prob[order[0]]:.1%})"); plt.show()
    return {CLASSES[i]: float(prob[i]) for i in order}

classify(test_df.path[0])

## 13. Export for the report / deliverables

In [ ]:
# All metrics in one JSON + the final table printed as Markdown (paste straight into the report)
export = {n: {k: (float(v) if isinstance(v, (float, np.floating)) else v)
              for k, v in r.items() if k in ("accuracy", "precision", "recall", "f1", "train_time_s", "infer_ms", "params", "best_params")}
          for n, r in RESULTS.items()}
json.dump(export, open(OUT_DIR / "metrics.json", "w"), indent=2, default=str)
json.dump({"classes": CLASSES, "train": len(train_df), "val": len(val_df), "test": len(test_df),
           "removed_duplicates": int(len(df_dups)), "corrupted": len(bad)},
          open(OUT_DIR / "dataset_stats.json", "w"), indent=2)
try:
    try:
    print(summary.round(4).to_markdown())
except ImportError:
    print(summary.round(4))
except ImportError:
    print(summary.round(4))
print("\nSaved:", sorted(p.name for p in OUT_DIR.iterdir()))

In [ ]:
# Zip the processed dataset for submission (deliverable: "dataset organised and zipped")
def zip_dataset(src=RAW_DIR, dst=OUT_DIR / "household_objects_dataset.zip"):
    with zipfile.ZipFile(dst, "w", zipfile.ZIP_DEFLATED) as z:
        for p in Path(src).rglob("*"):
            if p.is_file(): z.write(p, p.relative_to(Path(src).parent))
    print(f"Wrote {dst} ({dst.stat().st_size / 1e6:.1f} MB)")
# zip_dataset()   # uncomment when the dataset is final